# FEniCS showcase: FEM solvers as composable pbg processes

_Investigation `fenics-showcase` — coder reproduction notebook._

**Question.** Can modern FEniCSx be wrapped as composable pbg processes that reproduce
canonical FEM results and enable bigraph coupling?

A showcase investigation proving that a real external FEM solver
(FEniCSx/dolfinx, not a mock or reimplementation) can be wrapped as
process-bigraph Steps/Processes and composed like any other pbg model.
Two validation studies (`poisson-validation`, `mesh-convergence`) confirm
numerical fidelity against known analytic/theoretical results; two
dynamics studies (`transient-diffusion`, `reaction-diffusion`) extend the
bridge to time-stepping and cross-process coupling. Three ADVANCED
studies (`navier-stokes`, `moving-boundary`, `complex-geometry`) push
the same bridge into fluid dynamics, deforming domains, and
gmsh-generated non-rectangular geometry.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-fenics/viva-fenics').is_dir():
    REPO = Path('/home/runner/work/viva-fenics/viva-fenics')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_fenics.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `poisson-validation`

**Question.** Does the dolfinx-backed `PoissonSolverStep` reproduce the exact solution of
a manufactured steady Poisson problem to numerical precision?

**Objective.** Solve the steady Poisson equation -div(grad(u)) = f on the unit square with
a manufactured (known-exact) quadratic solution, and verify the FEM
solution matches the exact solution to within numerical tolerance. This is
the correctness baseline every other study in the investigation builds on.

**Hypothesis.** Degree-2 Lagrange elements exactly represent the quadratic manufactured
solution used here (Galerkin orthogonality), so the L2 error against the
analytic solution should sit at floating-point round-off, not merely
"small".


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: poisson-validation ===
STUDY = 'poisson-validation'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**solution-heatmap**


In [ ]:
# solution-heatmap
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| l2-error-within-tolerance | kind=derived_scalar field=l2_error | op < value 1e-10 provenance {'kind': 'theory', 'note': 'A quadratic manufactured solution lies exactly in the degree-2 Lagrange trial space, so the discretization error is zero up to solver/quadrature round-off; 1e-10 is a generous ceiling above that floor (observed ~2.8e-13).'} |


## Study: `mesh-convergence`

**Question.** Does the `PoissonSolverStep` wrapper deliver the theoretical O(h^2) mesh
convergence rate for P1 Lagrange elements, or does the wrapping introduce
an artifact that masks/inflates the true discretization error?

**Objective.** Sweep the `mesh_convergence` composite's `resolution` parameter (8, 16, 32)
at fixed polynomial degree 1, fit the L2-error-vs-mesh-size slope on a
log-log plot, and confirm the fitted rate matches the theoretical
O(h^(degree+1)) convergence order.

**Hypothesis.** On a superconvergence-resistant (crossed-diagonal) unit-square mesh, the
L2 error of a P1 (degree=1) solve should scale as O(h^2) across a
resolution sweep, giving an observed log-log slope near 2.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mesh-convergence ===
STUDY = 'mesh-convergence'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**convergence-loglog**


In [ ]:
# convergence-loglog
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| convergence-rate-matches-order | kind=derived_scalar field=convergence_rate | op >= value 1.7 provenance {'kind': 'theory', 'note': 'Standard FEM a priori error theory gives O(h^(k+1)) for degree-k Lagrange elements in the L2 norm; k=1 implies rate=2. 1.7 is a calibration margin below the theoretical value, not a fitted threshold.'} |


## Study: `transient-diffusion`

**Question.** Does a time-stepped `DiffusionProcess` (backward-Euler, dolfinx-backed)
conserve mass under a no-flux boundary while smoothing/decaying an initial
field, when driven through process-bigraph's own Composite tick loop
(not called directly)?

**Objective.** Run the `transient_diffusion` composite for several ticks and verify (a)
the field's peak decays as diffusion spreads the initial gaussian bump and
(b) the FEM-integral mass stays conserved (no-flux boundary), exercising
the additive-delta store convention through a real Composite.run(), not
just DiffusionProcess.update() in isolation.

**Hypothesis.** With no Dirichlet boundary condition (natural Neumann / zero-flux), the
FEM integral of the field should stay flat tick over tick while the
gaussian bump's peak monotonically decays as it spreads.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: transient-diffusion ===
STUDY = 'transient-diffusion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**field-animation**


In [ ]:
# field-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mass-conserved | kind=derived_scalar field=mass_drift_fraction | op < value 0.05 provenance {'kind': 'theory', 'note': 'Zero-flux (natural Neumann) boundary conditions conserve the domain integral exactly in the continuous problem; 5% is a numerical-noise margin for the discrete backward-Euler solve.'} |
| peak-decays | kind=derived_scalar field=peak_ratio | op < value 1.0 |


## Study: `reaction-diffusion`

**Question.** Do two independently-authored processes -- `DiffusionProcess` (no
knowledge of reactions) and `LogisticReactionProcess` (no knowledge of
FEM/diffusion) -- produce genuine Fisher-KPP reaction-diffusion dynamics
when wired to the same bigraph stores, with neither process calling into
the other?

**Objective.** Run the `reaction_diffusion` composite (DiffusionProcess ⊕
LogisticReactionProcess) and confirm (a) mass grows far faster than a
near-zero-reaction control under identical wiring/duration -- isolating
the reaction term's causal contribution from solver noise -- and (b) the
field stays bounded near the logistic carrying capacity K rather than
blowing up, which would indicate the `source` store were composing
additively instead of being overwritten each tick.

**Hypothesis.** Coupling is entirely a property of the document wiring (shared
`stores.solution` / `stores.source` paths), not of either process's
implementation: with the reaction term active (r=2.0), the field's total
mass should grow dramatically faster than a near-zero-reaction control
(r=1e-9, identical wiring) run for the same duration, while logistic
saturation keeps the field bounded near the carrying capacity K.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: reaction-diffusion ===
STUDY = 'reaction-diffusion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**wavefront-animation**


In [ ]:
# wavefront-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mass-grows-via-coupling | kind=derived_scalar field=growth_ratio | op >= value 5.0 provenance {'kind': 'calibration', 'note': '5x is a clear-separation band between genuine reaction-driven growth and residual solver/quadrature noise, calibrated against the r~=0 control (observed ratio ~2.6e9x at r=2.0 vs r=1e-9, D=0.05, resolution=24, run_time=0.5).'} |
| field-bounded-by-k | kind=derived_scalar field=max_field | op < value 1.2 provenance {'kind': 'theory', 'note': 'Logistic saturation caps u at K=1.0 in the continuous limit regardless of run length; 1.2 leaves headroom for discretization overshoot while still catching an accumulate-vs-overwrite regression, which blows well past this bound.'} |


## Study: `navier-stokes`

**Question.** Does a real dolfinx incompressible Navier-Stokes solve -- Taylor-Hood-ish
P2/P1 velocity/pressure, classic IPCS (Incremental Pressure Correction
Scheme) operator splitting -- reproduce the textbook lid-driven-cavity
flow (a non-trivial, approximately divergence-free, quasi-steady
recirculating flow pinned near the lid velocity) when wrapped as a
process-bigraph Process, and does the scheme stay numerically stable
across a moderate Reynolds-number sweep?

**Objective.** Run the `navier_stokes` composite (lid-driven cavity on the unit square,
P2 velocity / P1 pressure, IPCS splitting) to quasi-steady state and
verify (a) the flow is non-trivial and bounded near the lid velocity, (b)
the velocity field is approximately divergence-free, and (c) the same
scheme stays stable (finite, non-blown-up) across a Reynolds sweep
(100, 400, 1000).

**Hypothesis.** `NavierStokesProcess` should converge, within a fraction of a second of
simulated time, to a quasi-steady cavity flow whose peak speed sits near
the prescribed lid velocity (the moving-lid Dirichlet BC pins it there)
and whose velocity field is approximately divergence-free (small, not
exactly zero -- IPCS is only an approximate projection). The same IPCS
scheme, unmodified, should remain stable (no blow-up) from Re=100 up
through Re=1000 at this mesh/timestep, since diffusion is treated
implicitly (Crank-Nicolson) even though convection is explicit.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: navier-stokes ===
STUDY = 'navier-stokes'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cavity-flow-streamlines**


In [ ]:
# cavity-flow-streamlines
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**cavity-pressure-heatmap**


In [ ]:
# cavity-pressure-heatmap
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reaches-quasi-steady | kind=derived_scalar field=quasi_steady_rel_change | op < value 0.05 provenance {'kind': 'calibration', 'note': '5% is a generous convergence band above the observed residual change (~1e-3 or smaller) once the cavity flow has settled, calibrated during development at resolution=24, dt=0.01.'} |
| approximately-divergence-free | kind=derived_scalar field=mean_abs_divergence | op < value 0.05 provenance {'kind': 'theory', 'note': "IPCS operator splitting introduces an O(dt) divergence residual (plus a local O(h) artifact at the lid's two discontinuous-BC corners) that is not exactly zero but should be small and shrink under refinement; 0.05 is well above the ~0.003 observed at this resolution during development."} |
| stable-across-reynolds-sweep | kind=derived_scalar field=max_speed_across_sweep | op < value 2.0 provenance {'kind': 'calibration', 'note': 'The lid BC directly prescribes the max speed at u=lid_velocity; 2x leaves generous headroom for transient overshoot while still catching a genuinely unstable/blown-up solve. Verified stable (no reduction from the planned Re=1000 top variant was needed) during development.'} |


## Study: `moving-boundary`

**Question.** Does a real dolfinx diffusion solve, re-assembled and re-solved every
substep on a mesh whose geometry is mutated in place according to a
prescribed moving-boundary law (prescribed ALE mesh motion), stay
numerically well-behaved -- boundary tracking the prescribed law, domain
measure scaling correctly with the deformation amplitude, field staying
finite -- when wrapped as a process-bigraph Process?

**Objective.** Run the `moving_boundary` composite (a single `MovingBoundaryProcess` --
real dolfinx diffusion re-solved on a prescribed-ALE deforming mesh) and
verify (a) `boundary_position` tracks the prescribed oscillation law to
near machine precision, (b) `domain_measure`'s swing away from the base
area scales with the oscillation amplitude across an amplitude sweep, and
(c) the diffusion field stays finite and bounded across the run.

**Hypothesis.** `MovingBoundaryProcess` should reproduce the prescribed top-boundary law
`y_top(t)` to near machine precision (the mesh deformation is an exact
uniform vertical stretch, not an approximation), the FEM-assembled domain
area should scale linearly with the oscillation amplitude (a larger
amplitude produces a proportionally larger swing in domain measure away
from the base area), and the diffusion field solved on the deforming mesh
should remain finite and bounded throughout, since no term in the
backward-Euler diffusion assembly can blow up under a smooth, bounded
mesh deformation.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: moving-boundary ===
STUDY = 'moving-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**moving-boundary-animation**


In [ ]:
# moving-boundary-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| boundary-follows-prescribed-law | kind=derived_scalar field=boundary_law_abs_error | op < value 0.0001 provenance {'kind': 'theory', 'note': "The mesh motion is deform_mesh()'s exact uniform stretch by y_top(t); the only departure from exact equality is floating-point roundoff (observed ~1e-15 during development), so 1e-4 leaves enormous headroom while still catching a genuinely broken mesh motion (e.g. a stale/undeformed mesh)."} |
| domain-measure-scales-with-amplitude | kind=derived_scalar field=swing_ratio | op >= value 3.0 provenance {'kind': 'calibration', 'note': 'domain_measure = y_top(t) exactly, so the analytic swing ratio for a 5x amplitude ratio is exactly 5.0; 3.0 leaves headroom for transient/sampling effects (peak not landing exactly at the same sampled tick in both runs) while still catching a broken or amplitude-insensitive mesh motion.'} |
| field-stays-finite | kind=derived_scalar field=max_abs_field | op < value 5.0 provenance {'kind': 'theory', 'note': 'Backward-Euler diffusion of a bounded gaussian-bump initial condition (peak amplitude 1.0) with no source term cannot exceed its initial peak in the continuous limit; 5.0 leaves generous headroom for discretization overshoot on the deforming mesh while still catching a genuine numerical blow-up.'} |


## Study: `complex-geometry`

**Question.** Does a real gmsh-constructed non-trivial planar geometry -- imported into
dolfinx via `dolfinx.io.gmsh.model_to_mesh` -- produce a genuine
unstructured triangulation with a positive cell count, and does a real
Poisson solve on that imported mesh (homogeneous Dirichlet boundary, a
constant interior source) stay finite, bounded, and non-trivially
nonzero, across three qualitatively different domain topologies (a
simply-connected domain with a circular hole, a non-convex re-entrant
corner, and a multiply-connected annulus)?

**Objective.** Build the `complex_geometry` composite (a single `ComplexGeometryStep`:
gmsh geometry construction -> dolfinx import -> Poisson solve) for each of
the three named geometries (obstacle, L-shape, annulus) and verify (a)
each mesh imports with a positive, non-trivial cell count and (b) each
solve returns a finite, bounded, non-trivial solution field.

**Hypothesis.** The gmsh -> dolfinx import path is standard and robust for any of these
three domains, so each should import with a cell count well above a
trivial handful, and -Laplacian(u) = 1 with u=0 on the full boundary
should have a smooth, non-negative (maximum-principle), interior-peaked
solution on every one of them -- confirming the FEniCSx bridge works on
genuinely non-trivial, non-rectangular domains, not only the unit square
every other study in this investigation uses.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: complex-geometry ===
STUDY = 'complex-geometry'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**field-heatmap-obstacle**


In [ ]:
# field-heatmap-obstacle
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**mesh3d**


In [ ]:
# mesh3d
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mesh-imports-with-positive-cells | kind=derived_scalar field=min_n_cells_across_geometries | op > value 20 provenance {'kind': 'calibration', 'note': "Observed cell counts at resolution=32 run in the low thousands for all three geometries (see studies/complex-geometry/sims/run.py's printed summary); 20 is a generous floor that only fails on a genuinely degenerate/empty import, not on any real triangulation density."} |
| solution-finite-bounded-nontrivial | kind=derived_scalar field=min_solution_max_across_geometries | op > value 1e-06 provenance {'kind': 'theory', 'note': 'The maximum principle for -Laplacian(u)=const>0 with u=0 on the boundary guarantees a smooth, non-negative, interior-peaked solution on any of these domains; a solution that is all-zero, negative anywhere, or non-finite indicates a broken boundary condition or a corrupted imported mesh, not solver noise.'} |
| solution-stays-bounded | kind=derived_scalar field=max_solution_max_across_geometries | op < value 1.0 provenance {'kind': 'calibration', 'note': "Observed peak solution values at resolution=32 sit around 0.01-0.04 across all three geometries (see run.py's printed summary); 1.0 leaves two orders of magnitude of headroom while still catching a genuine assembly/orientation blow-up."} |
